In [3]:
import kaggle_benchmarks as kbench
import urllib.request
import tarfile
import json
import random
import os
import csv

# --------------------------------------------------------------------------------
# 1. LEAN DATA LOADER (Grabs 50 items total per bucket)
# --------------------------------------------------------------------------------
def load_data():
    if not os.path.exists("v4_atomic_dev.csv"):
        print("Downloading ATOMIC...")
        urllib.request.urlretrieve("https://homes.cs.washington.edu/~msap/atomic/data/atomic_data.tgz", "atomic.tgz")
        with tarfile.open("atomic.tgz", "r:gz") as tar:
            tar.extract("v4_atomic_dev.csv")

    b = {"intent": [], "need": [], "react": []}
    cols = [("intent", 6), ("need", 7), ("react", 8)]

    with open("v4_atomic_dev.csv", "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            try:
                for key, idx in cols:
                    # Load 50 items (20 for the test, 30 to act as wrong answers)
                    if len(b[key]) < 50: 
                        val = json.loads(row[idx])
                        if val and val[0] != "none":
                            b[key].append({"event": row[0], "val": val[0]})
                if all(len(v) == 50 for v in b.values()): break
            except: pass
    return b

# --------------------------------------------------------------------------------
# 2. EVALUATION (Runs exactly 20 tests per category)
# --------------------------------------------------------------------------------
def evaluate(llm, data, label, q_text):
    score = 0
    pool = [d["val"] for d in data[20:]] # Use the remaining 30 items as distractors
    
    # Loop runs exactly 20 times
    for i, d in enumerate(data[:20]):
        correct = d["val"]
        opts = [correct] + random.sample([p for p in pool if p != correct], 2)
        random.shuffle(opts)
        ans = chr(65 + opts.index(correct))

        prompt = f"Event: {d['event']}\n{q_text}\nA) {opts[0]}\nB) {opts[1]}\nC) {opts[2]}\nAnswer (A/B/C):"

        # Open isolated chat for each of the 20 tests
        with kbench.chats.new(f"{label}_test_{i}"):
            res = llm.prompt(prompt).strip().upper()
            
        if res.startswith(ans) or ans in res[:3]:
            score += 1
            
    final_score = score / 20.0
    print(f"Finished {label} (20 samples) -> Score: {final_score:.2f}")
    return final_score

# --------------------------------------------------------------------------------
# 3. KAGGLE TASK
# --------------------------------------------------------------------------------
@kbench.task(name="ATOMIC: 20 Samples Per Category")
def atomic_benchmark(llm) -> float:
    b = load_data()
    if not any(b.values()): return 0.0

    s1 = evaluate(llm, b["intent"], "Intent", "Why did PersonX do this?")
    s2 = evaluate(llm, b["need"], "Need", "What did PersonX need to do BEFORE this?")
    s3 = evaluate(llm, b["react"], "React", "How does PersonX feel AFTER this?")

    print(f"\n20-Sample Test Scores -> Intent: {s1:.2f} | Need: {s2:.2f} | React: {s3:.2f}")
    return (s1 + s2 + s3) / 3.0

# Run
atomic_benchmark.run(kbench.llm)


5-Sample Test Scores -> Intent: 0.20 | Need: 0.40 | React: 0.80


BokehModel(combine_events=True, render_bundle={'docs_json': {'1cceb876-ec4a-457f-a2d8-150d2babf6a4': {'version…